In [1]:
!pip install nltk gensim torch==2.2.2 transformers==4.38.2 datasets accelerate==0.21.0 'fsspec<=2026.2.0' scikit-learn

  Using cached transformers-4.38.2-py3-none-any.whl.metadata (130 kB)
  Using cached accelerate-0.21.0-py3-none-any.whl.metadata (17 kB)
  Using cached tokenizers-0.15.2-cp310-cp310-macosx_11_0_arm64.whl.metadata (6.7 kB)
Using cached transformers-4.38.2-py3-none-any.whl (8.5 MB)
Using cached accelerate-0.21.0-py3-none-any.whl (244 kB)
Using cached tokenizers-0.15.2-cp310-cp310-macosx_11_0_arm64.whl (2.4 MB)
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 0.27.0
    Uninstalling accelerate-0.27.0:
      Successfully uninstalled accelerate-0.27.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.40.0
    Uninstalling transformers-4.40.0:m╸━━━━━━━━━━━━━ 2/3 [transformers]
      Successfully uninstalled transformers-4.40.0━━━━━━━━━━━━ 2/3 [transformers]
  

In [1]:
import torch
import transformers

print(torch.__version__)
print(transformers.__version__)

2.2.2
4.38.2


/Users/sarveshkhanna/miniconda3/envs/csml/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from gensim.models import FastText
from pprint import pprint
import os, pickle, re, json, random, sys
from tqdm import tqdm
from gensim.models.keyedvectors  import KeyedVectors


In [3]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/sarveshkhanna/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
from gensim.models import KeyedVectors


In [5]:
from nltk.corpus import stopwords
sws_path = r'/Users/Nishu/Course/CSML/stopwords.pkl'

sws = set(stopwords.words('english'))
with open(sws_path, 'rb') as f:
	stopwords = pickle.load(f)


In [6]:
import re

def is_safe_token(token):
    # avoid numbers, short tokens, uppercase entities
    if len(token) < 3:
        return False
    if re.search(r'\d', token):
        return False
    if token.isupper():
        return False
    if token in stopwords:
        return False
    return True

In [7]:
from gensim.models import KeyedVectors

# Load binary model
model_w2v = KeyedVectors.load_word2vec_format("/Users/Nishu/Course/CSML/BioWordVec_PubMed_MIMICIII_d200.vec.bin", binary=True)

In [8]:
import random

def substitute_words(text, model, max_replacements=2, similarity_threshold=0.92):
    parts = re.split(r'(\W+)', text)
    candidate_indices = [
        i for i, part in enumerate(parts)
        if re.fullmatch(r'\w+', part)
        and part in model.key_to_index
        and is_safe_token(part)
    ]

    if not candidate_indices:
        return text

    num_replacements = min(max_replacements, len(candidate_indices))
    indices_to_replace = random.sample(candidate_indices, num_replacements)

    for idx in indices_to_replace:
        word = parts[idx]
        try:
            similar_words = model.most_similar(word, topn=20)
            filtered = [w for w, sim in similar_words if sim >= similarity_threshold]
            if filtered:
                parts[idx] = random.choice(filtered)
        except KeyError:
            continue

    return ''.join(parts)


In [9]:
def flatten_context(context):
    if isinstance(context, dict):
        all_parts = []

        for value in context.values():
            if isinstance(value, list):
                all_parts.append(" ".join(value))  # join sentences
            elif isinstance(value, str):
                all_parts.append(value)

        return " ".join(all_parts)

    return context

In [10]:

IMPORTANT_SECTIONS = ["results", "conclusions"]

# def augment_context(context, model):
#     if isinstance(context, dict):
#         new_context = context.copy()

#         for key in IMPORTANT_SECTIONS:
#             if key in context:
#                 section = context[key]

#                 if isinstance(section, list):
#                     # augment each sentence
#                     new_context[key] = [
#                         substitute_words(sent, model)
#                         for sent in section
#                     ]
#                 elif isinstance(section, str):
#                     new_context[key] = substitute_words(section, model)

#         return new_context
    
def augment_context(context, model):
    if isinstance(context, dict) and "contexts" in context:
        new_context = context.copy()
        new_context["contexts"] = [
            substitute_words(sent, model)
            for sent in context["contexts"]
        ]
        return new_context

    # fallback for other formats
    context_text = flatten_context(context)
    augmented_text = substitute_words(context_text, model)
    return augmented_text

    return substitute_words(context, model)

In [11]:
def build_dataset(data, model_w2v, augment=False, max_samples=8000):
    questions = []
    contexts = []
    labels = []

    label_map = {"yes": 0, "no": 1, "maybe": 2}

    for i, sample in enumerate(data):
        context = sample["context"]
        question = sample["question"]
        label = sample["final_decision"]

        if label not in label_map:
            continue

        if augment:
            context = augment_context(context, model_w2v)

        context_text = flatten_context(context)

        questions.append(question)
        contexts.append(context_text)
        labels.append(label_map[label])

        if augment and len(labels) >= max_samples:
            break

    return questions, contexts, labels


In [ ]:
# Datasets already installed

In [ ]:
!pip cache purge

In [12]:
from datasets import load_dataset

dataset = load_dataset("pubmed_qa", "pqa_labeled")
# unlabeled_dataset = load_dataset("pubmed_qa", "pqa_unlabeled")

data_original = dataset["train"]

# Initial split into original train and test sets
split_dataset = data_original.train_test_split(test_size=0.2, seed=42)

train_data = split_dataset["train"]
val_data = split_dataset["test"]

In [ ]:
# Original
train_questions, train_contexts, train_labels = build_dataset(train_data, model_w2v, augment=False)

# Augmented (small!)
aug_questions, aug_contexts, aug_labels = build_dataset(train_data, model_w2v, augment=True, max_samples=8000)

# Combine
train_questions += aug_questions
train_contexts += aug_contexts
train_labels += aug_labels

# Validation
val_questions, val_contexts, val_labels = build_dataset(val_data, model_w2v, augment=False)


In [15]:
print(len(train_questions), len(train_labels))
print(len(val_questions), len(val_labels))
print(train_questions[0])
print(train_contexts[0])


1600 1600
200 200
Is there a differential in the dental health of new recruits to the British Armed Forces?
Figures from the British Defence Dental Services reveal that serving personnel in the British Army have a persistently lower level of dental fitness than those in the Royal Navy or the Royal Air Force. No research had been undertaken to ascertain if this reflects the oral health of recruits joining each Service. This study aimed to pilot a process for collecting dental and sociodemographic data from new recruits to each Service and examine the null hypothesis that no differences in dental health existed. Diagnostic criteria were developed, a sample size calculated and data collected at the initial training establishments of each Service. Data for 432 participants were entered into the analysis. Recruits in the Army sample had a significantly greater prevalence of dental decay and greater treatment resource need than either of the other two Services. Army recruits had a mean numbe

In [16]:
import torch
import transformers
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.1")

train_encodings = tokenizer(
    train_questions,
    train_contexts,
    truncation=True,
    padding=True,
    max_length=512,
)
val_encodings = tokenizer(
    val_questions,
    val_contexts,
    truncation=True,
    padding=True,
    max_length=512,
)


/Users/sarveshkhanna/miniconda3/envs/csml/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [17]:
import torch

class QADataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = QADataset(train_encodings, train_labels)
val_dataset = QADataset(val_encodings, val_labels)

In [18]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.1",
    num_labels=3
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [19]:
from transformers import Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro")
    }

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    weight_decay=0.01,
    warmup_steps=100,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)


In [20]:
trainer.train()

  0%|          | 0/600 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [22]:
import pandas as pd
train_df = pd.DataFrame({
"question": train_questions,
"context": train_contexts,
"label": train_labels,
})

val_df = pd.DataFrame({
"question": val_questions,
"context": val_contexts,
"label": val_labels,
})

In [23]:
train_df.to_csv("pubmedqa_aug_train.csv", index=False)
val_df.to_csv("pubmedqa_val.csv", index=False)

